# L08 · Demonstration Acquisition and Scripted Experts

This lab applies the scripted-expert route selected in the lecture to one complete banana-to-bowl task:

```text
demonstration source choice → task and grasp contract → seven-phase expert
→ in-memory command/state trace → containment evidence
```

The notebook exposes the task contract, phase logic, command schedule, measured trace, and outcome test while reusing the shared controller implementation.

## Before you run

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD backend when available and otherwise uses CPU. Set it to `cpu` to require the minimum-compatible backend. `ROBO_GENESIS_RENDER=0` runs the complete rollout without creating a camera; set it to `1` before starting the kernel to require eight world-view stage images. Restart the kernel before changing either setting.

The CPU path is a valid fallback, not a recommendation to prefer CPU; it generally has lower simulation throughput than the verified AMD path.

Predict first:

1. Which information belongs to `TaskSpec`, and which belongs to `GraspProfile`?
2. Why does settling occur before the seven action phases?
3. Why can commanded action and measured state both have shape `(T, 9)` without being equal?
4. Why must bowl containment check both horizontal placement and below-rim depth?

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.build_scene import build_scene
from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.grasp_demo import (
    GraspProfile,
    MOVE_MAX_DQ,
    MOVE_MIN_STEPS,
    TaskSpec,
    check_success,
    run_pick_place,
)
from robo_genesis.scene_config import WORLD_CAM_RES

lesson = load_course_manifest().lesson('L08')
assert lesson.slug == 'demonstration-acquisition-and-scripted-experts'
assert lesson.duration_minutes == 120
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'planned'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '0').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'

runtime = notebook_mode('l08-scripted-expert', show_viewer=False)
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == 'cpu' else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision='32', logging_level='warning')

if getattr(gs, 'amdgpu', None) is not None and gs.backend == gs.amdgpu:
    actual_backend = 'amdgpu'
elif gs.backend == gs.cpu:
    actual_backend = 'cpu'
else:
    actual_backend = str(gs.backend)

print('Genesis:', environment['genesis_world'])
print('requested backend:', backend_mode)
print('actual backend:', actual_backend)
print('render enabled:', render_enabled)
print('output directory:', runtime['output_dir'].resolve())


## Make the expert contract explicit

`TaskSpec` states what should happen: pick the banana, place it in the bowl, and use a horizontal tolerance. The selected `GraspProfile` states how this object is grasped: jaw yaw, hand height strategy, and closing force.

The compact phase table below is the readable state-machine skeleton. The shared `run_pick_place()` implementation remains the behavior source of truth.

In [ ]:
PHASES = (
    ('pregrasp', 'pose above object', 'IK + collision-checked plan'),
    ('descend', 'fixed-xy vertical approach', 'IK at descending z waypoints'),
    ('grasp', 'establish and hold contact', 'arm position + finger force'),
    ('lift', 'clear the tabletop', 'bounded-increment arm targets'),
    ('transport', 'move above the bowl', 'bounded-increment arm targets'),
    ('release', 'let the object enter the bowl', 'open finger position target'),
    ('retreat', 'move hand away and settle', 'direct retreat + settle'),
)
EXPECTED_FRAME_TAGS = (
    '00_start',
    '01_pregrasp',
    '02_reach',
    '03_grasp',
    '04_lift',
    '05_above_target',
    '06_release',
    '07_done',
)

task = TaskSpec(
    pick_object='011_banana',
    place_target='024_bowl',
    success_tol=0.06,
)
profile = task.grasp_profile()
contract_checks = {
    'seven_ordered_phases': tuple(row[0] for row in PHASES)
    == ('pregrasp', 'descend', 'grasp', 'lift', 'transport', 'release', 'retreat'),
    'banana_to_bowl_task': task.pick_object == '011_banana'
    and task.place_target == '024_bowl',
    'grasp_profile_type': isinstance(profile, GraspProfile),
    'finite_profile': np.isfinite(
        [profile.yaw_offset, profile.grasp_hand_z, profile.close_force]
    ).all(),
}

for index, (phase, target, control) in enumerate(PHASES, start=1):
    print(f'{index}. {phase:<10} target={target:<31} control={control}')
print('task:', task)
print('profile:', profile)
for name, passed in contract_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(contract_checks.values()):
    raise AssertionError(contract_checks)


## Verify the commanded-target schedule

During lift and transport, the expert linearly interpolates from measured `q_start` to an IK goal. The number of waypoints is `max(MOVE_MIN_STEPS, ceil(Δq∞ / MOVE_MAX_DQ))`, so the infinity-norm change between adjacent commanded arm targets is bounded.

The second calculation is the one-variable exercise: predict what happens when `CANDIDATE_MAX_DQ` is smaller, then compare the waypoint count and maximum command step. This checks the command schedule only; it does not claim a particular measured acceleration or a higher grasp-success rate.

In [ ]:
q_start = np.array([0.00, -0.30, 0.10, -1.80, 0.05, 1.55, 0.70], dtype=float)
q_goal = np.array([0.18, -0.08, 0.32, -1.42, -0.11, 1.82, 0.54], dtype=float)

delta_q_inf = float(np.max(np.abs(q_goal - q_start)))
waypoint_count = max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / MOVE_MAX_DQ)))
fractions = np.arange(1, waypoint_count + 1, dtype=float)[:, None] / waypoint_count
command_schedule = q_start + (q_goal - q_start) * fractions
schedule_with_start = np.vstack([q_start, command_schedule])
max_command_step = float(np.max(np.abs(np.diff(schedule_with_start, axis=0))))

CANDIDATE_MAX_DQ = 0.003
candidate_waypoint_count = max(
    MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / CANDIDATE_MAX_DQ))
)
candidate_fractions = (
    np.arange(1, candidate_waypoint_count + 1, dtype=float)[:, None]
    / candidate_waypoint_count
)
candidate_schedule = q_start + (q_goal - q_start) * candidate_fractions
candidate_with_start = np.vstack([q_start, candidate_schedule])
candidate_max_step = float(np.max(np.abs(np.diff(candidate_with_start, axis=0))))

schedule_checks = {
    'waypoint_formula': waypoint_count
    == max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / MOVE_MAX_DQ))),
    'baseline_endpoint': np.allclose(command_schedule[-1], q_goal),
    'baseline_step_bound': max_command_step <= MOVE_MAX_DQ + 1e-12,
    'candidate_endpoint': np.allclose(candidate_schedule[-1], q_goal),
    'candidate_step_bound': candidate_max_step <= CANDIDATE_MAX_DQ + 1e-12,
    'smaller_bound_uses_no_fewer_waypoints': candidate_waypoint_count >= waypoint_count,
}

print(f'Δq∞: {delta_q_inf:.6f} rad')
print(
    f'baseline max_dq={MOVE_MAX_DQ:.6f}: '
    f'{waypoint_count} waypoints, max step={max_command_step:.6f} rad'
)
print(
    f'candidate max_dq={CANDIDATE_MAX_DQ:.6f}: '
    f'{candidate_waypoint_count} waypoints, max step={candidate_max_step:.6f} rad'
)
for name, passed in schedule_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(schedule_checks.values()):
    raise AssertionError(schedule_checks)


## Build the fixed task scene

The shared builder reuses the L07 tabletop, Franka, YCB objects, controller configuration, and stable asset paths. This lesson keeps the task fixed and passes `scene_dr=None`; domain randomization belongs to L11.

With rendering disabled, no camera is created. With rendering enabled, only the fixed world camera is requested because the goal here is a stage montage rather than a new camera lesson.

In [ ]:
bundle = build_scene(
    show_viewer=False,
    n_envs=1,
    add_world_cam=render_enabled,
    add_wrist_cam=False,
    add_video_cam=False,
    draw_world_frame=False,
    scene_dr=None,
)
initial_qpos = to_numpy(bundle.franka.get_qpos()).astype(float).reshape(-1)

build_checks = {
    'unbatched_franka_qpos': initial_qpos.shape == (9,),
    'franka_qpos_finite': np.isfinite(initial_qpos).all(),
    'task_entities_present': task.pick_object in bundle.ycb
    and task.place_target in bundle.ycb,
    'requested_world_camera': (bundle.world_cam is not None) == render_enabled,
    'no_wrist_camera': bundle.wrist_cam is None,
    'no_video_camera': bundle.video_cam is None,
}
for name, passed in build_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(build_checks.values()):
    raise AssertionError(build_checks)

print('Franka qpos shape:', initial_qpos.shape)
print('task entities:', task.pick_object, '→', task.place_target)
print('world camera:', 'created' if render_enabled else 'SKIP — not created')


## Run one scripted rollout and keep the trace in memory

The lightweight recorder reads measured Franka qpos immediately before the corresponding command is sent and the simulator advances. It stores no files and defines no dataset schema:

```text
state_t  = measured [arm_q(7), finger_q(2)]
action_t = commanded [arm_target(7), finger_target(2)]
```

L09 will add timestamps, image sampling, episode boundaries, and persistent storage. This cell performs exactly one fixed rollout.

In [ ]:
class TraceRecorder:
    def __init__(self, scene_bundle):
        self.bundle = scene_bundle
        self.states = []
        self.actions = []

    def on_step(self, action):
        measured_qpos = to_numpy(self.bundle.franka.get_qpos()).astype(float).reshape(-1)
        commanded_action = to_numpy(action).astype(float).reshape(-1)
        self.states.append(measured_qpos.copy())
        self.actions.append(commanded_action.copy())


trace = TraceRecorder(bundle)
rollout_success, stage_frames = run_pick_place(
    bundle,
    task,
    save_frames=render_enabled,
    recorder=trace,
)
state_trace = np.stack(trace.states)
action_trace = np.stack(trace.actions)

print('rollout success:', rollout_success)
print('state trace:', state_trace.shape, state_trace.dtype)
print('action trace:', action_trace.shape, action_trace.dtype)
print('captured stage frames:', len(stage_frames))


## Separate trace evidence from task evidence

Matching `(T, 9)` arrays establish alignment and shape, not equality: finite controller response and contact can separate measured qpos from a requested target.

For the bowl outcome, the horizontal component asks whether the banana lies inside the allowed opening radius. The vertical component asks whether the banana AABB bottom has dropped at least the current 1 cm margin below the bowl rim. Both must pass, and their conjunction must agree with the shared `check_success()` predicate.

In [ ]:
trace_delta = np.abs(action_trace - state_trace)
trace_checks = {
    'same_nonzero_length': len(state_trace) == len(action_trace) > 0,
    'state_shape': state_trace.ndim == 2 and state_trace.shape[1] == 9,
    'action_shape': action_trace.ndim == 2 and action_trace.shape[1] == 9,
    'floating_arrays': np.issubdtype(state_trace.dtype, np.floating)
    and np.issubdtype(action_trace.dtype, np.floating),
    'finite_arrays': np.isfinite(state_trace).all() and np.isfinite(action_trace).all(),
    'command_state_difference_observed': bool(np.max(trace_delta) > 0.0),
}

banana = bundle.ycb[task.pick_object]
bowl = bundle.ycb[task.place_target]
banana_pos = to_numpy(banana.get_pos()).astype(float).reshape(-1)
bowl_pos = to_numpy(bowl.get_pos()).astype(float).reshape(-1)
banana_aabb = to_numpy(banana.get_AABB()).astype(float).reshape(2, 3)
bowl_aabb = to_numpy(bowl.get_AABB()).astype(float).reshape(2, 3)

horizontal_distance = float(np.linalg.norm(banana_pos[:2] - bowl_pos[:2]))
bowl_rim_radius = 0.5 * float(
    min(bowl_aabb[1, 0] - bowl_aabb[0, 0], bowl_aabb[1, 1] - bowl_aabb[0, 1])
)
allowed_radius = min(task.success_tol, bowl_rim_radius)
within_footprint = horizontal_distance < allowed_radius
BOWL_RIM_MARGIN = 0.01
object_aabb_bottom_z = float(banana_aabb[0, 2])
bowl_rim_z = float(bowl_aabb[1, 2])
inside_bowl = object_aabb_bottom_z < bowl_rim_z - BOWL_RIM_MARGIN
shared_success = check_success(bundle, task)

outcome_checks = {
    'within_footprint': within_footprint,
    'inside_bowl': inside_bowl,
    'expanded_predicate_matches_shared_check':
    bool(within_footprint and inside_bowl) == shared_success,
    'rollout_result_matches_shared_check': bool(rollout_success) == shared_success,
    'task_completed': bool(rollout_success),
}

print(f'horizontal distance: {horizontal_distance:.6f} m')
print(f'allowed radius:      {allowed_radius:.6f} m')
print(f'object AABB bottom:  {object_aabb_bottom_z:.6f} m')
print(f'bowl rim:            {bowl_rim_z:.6f} m')
for name, passed in {**trace_checks, **outcome_checks}.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")


## Inspect the optional phase montage

When rendering is enabled, the next cell validates and displays the settled start plus one world-view image after each phase. The tags describe observation points in the shared implementation: `02_reach` follows descend, `05_above_target` follows transport, and `07_done` follows retreat and settling.

When rendering is disabled, the cell verifies that no camera or frame was created and reports `SKIP`. Images help interpret the sequence, but the trace and containment checks decide whether the notebook passes.

## Checkpoint and extensions

Before the final cell, explain the evidence chain in your own words:

1. How do `TaskSpec` and `GraspProfile` divide task intent from grasp assumptions?
2. Which primitive and control mode does each phase use?
3. What does the rate-schedule calculation prove, and what physical behavior does it leave unproven?
4. If only `within_footprint` fails, which part of the outcome should you diagnose first?

For an optional follow-up, try a lemon or plum, multiple independent seeds, or the separate `motion_probe.py --compare` tool. Do not interpret this single successful rollout as an expert success rate.

In [ ]:
visual_status = 'SKIP — ROBO_GENESIS_RENDER=0; no camera or stage frames were created'
visual_checks = {
    'camera_absent_when_disabled': not render_enabled and bundle.world_cam is None,
    'frames_absent_when_disabled': not render_enabled and len(stage_frames) == 0,
}

if render_enabled:
    frame_tags = tuple(tag for tag, _ in stage_frames)
    frame_images = [to_numpy(image) for _, image in stage_frames]
    width, height = WORLD_CAM_RES
    visual_checks = {
        'world_camera_present': bundle.world_cam is not None,
        'eight_stage_frames': len(frame_images) == 8,
        'ordered_stage_tags': frame_tags == EXPECTED_FRAME_TAGS,
        'rgb_shapes': all(image.shape == (height, width, 3) for image in frame_images),
        'rgb_dtype': all(image.dtype == np.uint8 for image in frame_images),
        'finite_pixels': all(np.isfinite(image).all() for image in frame_images),
        'within_frame_variation': all(bool(np.std(image) > 0.0) for image in frame_images),
        'sequence_pixel_change': any(
            not np.array_equal(left, right)
            for left, right in zip(frame_images, frame_images[1:])
        ),
    }
    if all(visual_checks.values()):
        figure, axes = plt.subplots(2, 4, figsize=(16, 8))
        for axis, tag, image in zip(axes.ravel(), frame_tags, frame_images):
            axis.imshow(image)
            axis.set_title(tag)
            axis.axis('off')
        figure.tight_layout()
        plt.show()
        visual_status = 'PASSED — start + seven phase images validated'

final_checks = {
    'runtime_contract': environment['genesis_world'] == '1.3.3'
    and actual_backend in {'cpu', 'amdgpu'}
    and (backend_mode != 'cpu' or actual_backend == 'cpu'),
    'manifest_contract': lesson.status.value == 'planned'
    and lesson.duration_minutes == 120,
    'expert_contract': all(contract_checks.values()),
    'rate_schedule': all(schedule_checks.values()),
    'scene_build': all(build_checks.values()),
    'trace_evidence': all(trace_checks.values()),
    'outcome_evidence': all(outcome_checks.values()),
    'visual_branch': all(visual_checks.values()),
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError('L08 checks failed: ' + ', '.join(failed))

print('visual evidence:', visual_status)
print('L08 CHECK: PASSED')
